# The RAG Pipeline

This notebook builds the pipeline of the book section *The RAG Pipeline*, one stage at a
time, over a small knowledge base of Wikipedia articles about the Nobel Prizes. We start with
the reason to retrieve at all, index the articles at two levels, run sparse and dense
retrieval, fuse their rankings, rerank with a cross-encoder, and hand the top passages to a
language model that answers with citations. At the end we assemble the same pipeline in
Haystack, the framework of the book.

**Learning goals:**
- See the time limit of a language model's parametric knowledge, and how retrieval fixes it
- Index documents at two levels: small children for dense matching, larger parents for the generator
- Resolve dense hits to their parents and fuse them with BM25 by reciprocal rank fusion
- Rerank candidates with a cross-encoder and build a grounded, citing prompt
- Compare pipeline variants by quality and latency, and build the pipeline in Haystack

**Prerequisites:** ch07-01 (chunking), Chapter 4 (BM25), Chapter 5 (bi- and cross-encoders), Chapter 6 (vector search)

## Setup

In [1]:
import os
import time

os.environ["HAYSTACK_TELEMETRY_ENABLED"] = "False"   # Haystack sends usage statistics otherwise
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"

import logging
import warnings
warnings.filterwarnings("ignore")
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

from shared.display import print_table, display_md
from shared.collections import load_collection
from ch07_rag import (
    build_units, label, SparseIndex, DenseIndex, resolve_to_parents, rrf, rerank,
    build_prompt, ask, CACHED,
)

The generator is Claude on AWS Bedrock. Without AWS credentials, every generation cell shows an
answer recorded earlier instead, marked as such; everything else runs locally on a CPU.

## Why Retrieve at All?

A language model knows what it read during training, and nothing after. We ask it the
chapter's running question twice: for 2025 and for 2026.

In [2]:
rows = []
for year in (2025, 2026):
    question = f"Who won the Nobel Prize in Physics in {year}?"
    answer, live = ask(question, CACHED[f"memory-{year}"], max_tokens=300)
    rows.append([question, answer.strip() + ("" if live else " *(cached)*")])
print_table(rows, ["Question", "Answer from the model's memory"])

| Question                                    | Answer from the model's memory                                                                                                                                                                                                                                                                                                              |
|:--------------------------------------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Who won the Nobel Prize in Physics in 2025? | The 2025 Nobel Prize in Physics was awarded to **John Clarke, Michel H. Devoret, and John M. Martinis** "for the discovery of macroscopic quantum mechanical tunnelling and energy quantisation in an electric circuit."                                                                                                                    |
|                                             |                                                                                                                                                                                                                                                                                                                                             |
|                                             | Their experiments, carried out in the mid-1980s with superconducting circuits (Josephson junctions), showed that quantum effects such as tunnelling and discrete energy levels can appear in a system large enough to hold in your hand, not just at the scale of individual particles. This work underpins much of today's superconducting |
| Who won the Nobel Prize in Physics in 2026? | I don't know who won the 2026 Nobel Prize in Physics. My training data may not extend that far, and I don't have a reliable record of the announcement, which would normally come in early October 2026. I'd rather not guess.                                                                                                              |
|                                             |                                                                                                                                                                                                                                                                                                                                             |
|                                             | The most recent prize I'm fairly confident about is 2025, awarded to John Clarke, Michel Devoret, and John Martinis for their work on macroscopic quantum tunneling and energy quantization in electrical circuits.                                                                                                                         |

The 2025 prize, announced in October 2025, is already part of the model's training data, so it
answers correctly from memory. For that question RAG adds nothing; it is the "stable knowledge"
case of *When Not to Use RAG*. The 2026 prize lies past the training cutoff, and the model
rightly declines. This is the time limit of parametric knowledge: the boundary moves with every
new model, but there is always a boundary. From here on we use the 2026 question.

In [3]:
QUESTION = "Who won the Nobel Prize in Physics in 2026?"

## The Knowledge Base

The `nobel-2025` collection holds 38 Wikipedia articles, each pinned to a fixed revision so
that everyone retrieves the same text: the 2025 and 2026 prizes, the 2025 physics laureates,
their research and universities, the other 2025 laureates, the 2024 physics laureates, and
general articles about the Nobel Prize. Most of it is about the right prize in the wrong year,
or the right year in the wrong field: plenty of near misses for a retriever. The first load
downloads the articles (about 30 seconds) and caches them.

In [4]:
collection = load_collection("nobel-2025")
articles = sorted({d["title"] for d in collection})
display_md(
    f"**{len(articles)} articles, {len(collection)} sections, "
    f"{sum(len(d['text']) for d in collection):,} characters.**\n\n"
    + ", ".join(articles)
)

**38 articles, 416 sections, 733,595 characters.**

2025 Nobel Peace Prize, 2025 Nobel Prize in Literature, 2025 Nobel Prizes, 2026 Nobel Prizes, Alfred Nobel, Francis Halzen, Fred Ramsdell, Geoffrey Hinton, Google Quantum AI, IceCube Neutrino Observatory, Joel Mokyr, John Clarke (physicist), John Hopfield, John M. Martinis, Josephson effect, List of Nobel laureates in Physics, László Krasznahorkai, Mary E. Brunkow, María Corina Machado, Metal–organic framework, Michel Devoret, Nobel Prize, Nobel Prize in Physics, Nobel Prize medal, Omar M. Yaghi, Peter Howitt (economist), Philippe Aghion, Quantum tunnelling, Regulatory T cell, Richard Robson (chemist), Royal Swedish Academy of Sciences, Shimon Sakaguchi, Superconducting quantum computing, Susumu Kitagawa, Transmon, University of California, Berkeley, University of California, Santa Barbara, Yale University

Each retrieval unit is one section of an article. Tables are kept as rows of cells, because the
prize tables carry the facts:

In [5]:
unit = collection["2026 Nobel Prizes#Prizes"]
display_md(f"**{unit['id']}** ([revision]({unit['url']}))\n\n```\n{unit['text']}\n```")

**2026 Nobel Prizes#Prizes** ([revision](https://en.wikipedia.org/w/index.php?oldid=1380470315))

```
Prize | Recipient(s) | Reason
Physics | Francis Halzen | "for decisive contributions to the IceCube Neutrino Observatory and the discovery of high-energy neutrinos of astrophysical origin"
Physiology or Medicine | Karl Deisseroth | "for their discoveries concerning light-gated ion channels and optogenetics ".
Georg Nagel
Peter Hegemann
Peace | Navi Pillay | "for her efforts to promote peace and international law."
Literature | Anne Carson | "for her bold and inventive oeuvre that, in playful dialogue with the classical tradition, has created new forms for contemporary literature".
Chemistry | Henri B. Kagan | "for the discovery of non-linear effects and autocatalysis in asymmetric organic synthesis."
Kensō Soai
Economic Sciences | Robert Hall | "for microeconomic foundations of macroeconomic stabilisation."
Michael Woodford
```

## Indexing at Two Levels

Sections vary from a single line to over 20,000 characters. We split each section into
**parents** of at most 2000 characters, the units handed to the generator, and each parent into
**children** of about 400 characters, the units that drive dense retrieval. Every chunk starts
with its article and section name, so a child that only says "he joined Yale in 2002" still
tells the retriever who "he" is.

In [6]:
parents, children = build_units(collection, parent_size=2000, child_size=400)

display_md(
    f"| Level | Units | Mean characters |\n|---|---|---|\n"
    f"| Sections | {len(collection)} | {sum(len(d['text']) for d in collection) / len(collection):.0f} |\n"
    f"| Parents | {len(parents)} | {sum(len(p['text']) for p in parents) / len(parents):.0f} |\n"
    f"| Children | {len(children)} | {sum(len(c['text']) for c in children) / len(children):.0f} |"
)

| Level | Units | Mean characters |
|---|---|---|
| Sections | 416 | 1763 |
| Parents | 634 | 1200 |
| Children | 2682 | 319 |

In [7]:
parent = next(p for p in parents if p["title"] == "Francis Halzen" and "Nobel" in p["section"])
kids = [c for c in children if c["parent"] == parent["id"]]
rows = [[c["id"], len(c["text"]), c["text"][:110] + "..."] for c in kids]
display_md(f"**Parent {parent['id']}** ({label(parent)}, {len(parent['text'])} characters) "
           f"has {len(kids)} children:")
print_table(rows, ["Child", "Characters", "Text"])

**Parent 513** (Francis Halzen > Nobel Prize in Physics, 587 characters) has 2 children:

|   Child |   Characters | Text                                                                                                              |
|--------:|-------------:|:------------------------------------------------------------------------------------------------------------------|
|    2193 |          393 | Francis Halzen > Nobel Prize in Physics: In 2026, Halzen was awarded the Nobel Prize in Physics "for decisive ... |
|    2194 |          235 | Francis Halzen > Nobel Prize in Physics: . It explained that high-energy neutrinos can point us to distant ast... |

BM25 indexes the parents: term matching works on text of any length. The bi-encoder embeds the
children: all-MiniLM-L6-v2 reads at most 256 tokens, about 1000 characters, so a 2000-character
parent would be cut in half before it is embedded (see ch07-01).

In [8]:
EMBED_MODEL = "all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

bi_encoder = SentenceTransformer(EMBED_MODEL)
cross_encoder = CrossEncoder(RERANK_MODEL)

t0 = time.perf_counter()
sparse = SparseIndex(parents)
t1 = time.perf_counter()
dense = DenseIndex(children, bi_encoder)
t2 = time.perf_counter()
display_md(f"BM25 index over {len(parents)} parents: {t1 - t0:.1f} s. "
           f"Embeddings of {len(children)} children: {t2 - t1:.1f} s.")

BM25 index over 634 parents: 0.5 s. Embeddings of 2682 children: 12.9 s.

## Sparse and Dense Retrieval

Both engines search for the question. BM25 returns parents; the dense engine returns children.

In [9]:
sparse_hits = sparse.search(QUESTION, k=50)
child_hits = dense.search(QUESTION, k=50)

print_table(
    [[r + 1, label(parents[p]), f"{s:.2f}"] for r, (p, s) in enumerate(sparse_hits[:5])],
    ["Rank", "BM25: parent", "Score"])
print_table(
    [[r + 1, c, label(parents[children[c]["parent"]]), f"{s:.3f}"]
     for r, (c, s) in enumerate(child_hits[:5])],
    ["Rank", "Dense: child", "Its parent", "Cosine"])

|   Rank | BM25: parent                                    |   Score |
|-------:|:------------------------------------------------|--------:|
|      1 | Nobel Prize in Physics > Introduction           |    8.29 |
|      2 | Francis Halzen > Nobel Prize in Physics         |    7.97 |
|      3 | Francis Halzen > Honours and awards             |    7.87 |
|      4 | 2026 Nobel Prizes > Introduction                |    7.58 |
|      5 | List of Nobel laureates in Physics > Statistics |    7.43 |

|   Rank |   Dense: child | Its parent                                                |   Cosine |
|-------:|---------------:|:----------------------------------------------------------|---------:|
|      1 |              8 | Nobel Prize in Physics > Introduction                     |    0.734 |
|      2 |             33 | List of Nobel laureates in Physics > Statistics           |    0.721 |
|      3 |             22 | Nobel Prize in Physics > Nomination and selection         |    0.711 |
|      4 |           1211 | University of California, Santa Barbara > Notable faculty |    0.699 |
|      5 |             78 | List of Nobel laureates in Physics > Laureates            |    0.694 |

The two engines fail in opposite ways, as the book predicts. BM25 needs the exact token "2026",
and that is enough to bring Halzen's passages near the top, among general pages that repeat
"Nobel Prize in Physics". The dense engine matches the meaning of "who won the physics Nobel"
well, but the year dissolves in the embedding: its top hits are about the prize in general, in
any year. And several of its children belong to the same parent: the same passage, found
several times.

### Bringing both lists to the same unit

Rank fusion assumes both lists rank the same kind of item. We resolve each child to its parent
and keep only the best-ranked hit per parent.

In [10]:
dense_hits = resolve_to_parents(child_hits, children)
display_md(f"The dense engine's **{len(child_hits)} children** resolve to "
           f"**{len(dense_hits)} distinct parents**.")
print_table(
    [[r + 1, label(parents[p]), f"{s:.3f}"] for r, (p, s) in enumerate(dense_hits[:5])],
    ["Rank", "Dense: parent", "Best child cosine"])

The dense engine's **50 children** resolve to **24 distinct parents**.

|   Rank | Dense: parent                                             |   Best child cosine |
|-------:|:----------------------------------------------------------|--------------------:|
|      1 | Nobel Prize in Physics > Introduction                     |               0.734 |
|      2 | List of Nobel laureates in Physics > Statistics           |               0.721 |
|      3 | Nobel Prize in Physics > Nomination and selection         |               0.711 |
|      4 | University of California, Santa Barbara > Notable faculty |               0.699 |
|      5 | List of Nobel laureates in Physics > Laureates            |               0.694 |

## Hybrid Retrieval with Reciprocal Rank Fusion

Now both lists rank parents. Reciprocal rank fusion scores a parent by
$\sum_{\text{lists}} 1 / (60 + \text{rank})$: it ignores the incomparable raw scores of BM25
and cosine, and rewards parents that rank high in both lists.

In [11]:
fused = rrf([sparse_hits, dense_hits])
sparse_rank = {p: r + 1 for r, (p, _) in enumerate(sparse_hits)}
dense_rank = {p: r + 1 for r, (p, _) in enumerate(dense_hits)}

print_table(
    [[r + 1, label(parents[p]), sparse_rank.get(p, "-"), dense_rank.get(p, "-"), f"{s:.4f}"]
     for r, (p, s) in enumerate(fused[:8])],
    ["Rank", "Parent", "BM25 rank", "Dense rank", "RRF score"])
display_md(f"The candidate pool holds **{len(fused)} parents** from both engines.")

|   Rank | Parent                                                    |   BM25 rank |   Dense rank |   RRF score |
|-------:|:----------------------------------------------------------|------------:|-------------:|------------:|
|      1 | Nobel Prize in Physics > Introduction                     |           1 |            1 |      0.0328 |
|      2 | List of Nobel laureates in Physics > Statistics           |           5 |            2 |      0.0315 |
|      3 | List of Nobel laureates in Physics > Laureates            |          10 |            7 |      0.0292 |
|      4 | University of California, Santa Barbara > Notable faculty |          15 |            4 |      0.029  |
|      5 | 2026 Nobel Prizes > Prizes                                |           9 |           15 |      0.0278 |
|      6 | Nobel Prize in Physics > Nomination and selection         |          17 |            8 |      0.0277 |
|      7 | Nobel Prize in Physics > Nomination and selection         |          26 |            3 |      0.0275 |
|      8 | List of Nobel laureates in Physics > Introduction         |          19 |           12 |      0.0265 |

The candidate pool holds **61 parents** from both engines.

Fusion favours agreement. The general pages that both engines rank high now lead the list.
Halzen's own passage, high in the BM25 list but low in the dense list, falls back: it is still
in the pool, but no longer near the top. The pool is wider than either list; picking the best
passage from it is the next stage's job.

## Reranking with a Cross-Encoder

The bi-encoder embedded query and passage separately. A cross-encoder reads them together and
scores how well the passage answers the question. It is too slow for the whole collection but
affordable for the top 30 candidates.

In [12]:
t0 = time.perf_counter()
top = rerank(QUESTION, fused[:30], parents, cross_encoder, top_k=5)
rerank_ms = 1000 * (time.perf_counter() - t0)

fused_rank = {p: r + 1 for r, (p, _) in enumerate(fused)}
print_table(
    [[r + 1, label(parents[p]), fused_rank[p], f"{s:.2f}", parents[p]["body"][:90] + "..."]
     for r, (p, s) in enumerate(top)],
    ["Rank", "Parent", "Rank after RRF", "Cross-encoder", "Starts"])
display_md(f"Reranking 30 candidates took {rerank_ms:.0f} ms.")

|   Rank | Parent                                          |   Rank after RRF |   Cross-encoder | Starts                                                                                        |
|-------:|:------------------------------------------------|-----------------:|----------------:|:----------------------------------------------------------------------------------------------|
|      1 | Francis Halzen > Nobel Prize in Physics         |               14 |            9.54 | In 2026, Halzen was awarded the Nobel Prize in Physics "for decisive contributions to the ... |
|      2 | List of Nobel laureates in Physics > Statistics |                2 |            8.6  | The Nobel Prize in Physics has been awarded to 230 individuals as of 2026. The first prize... |
|      3 | List of Nobel laureates in Physics > Laureates  |                3 |            7.64 | 2025 |  | John Clarke | United Kingdom | "For the discovery of macroscopic quantum mechani... |
|      4 | Geoffrey Hinton > Honours and awards            |               26 |            7.41 | In 2024, he was jointly awarded the Nobel Prize in Physics with John Hopfield "for foundat... |
|      5 | 2026 Nobel Prizes > Prizes                      |                5 |            6.83 | Prize | Recipient(s) | Reason                                                                 |
|        |                                                 |                  |                 | Physics | Francis Halzen | "for decisive contributions to th...                               |

Reranking 30 candidates took 584 ms.

The cross-encoder lifts Halzen's passage from deep in the fused list to rank 1, and keeps the
prize table that names him in the top 5. Some general passages stay: they do mention who won
the prize, only for other years. The generator must sort that out, guided by the prompt.

## Generating a Grounded Answer

The five reranked parents go into the book's prompt skeleton: task and scope first, the
passages in a tagged block with numbers, the question, and a closing grounding reminder.

In [13]:
prompt = build_prompt(QUESTION, [p for p, _ in top], parents)
display_md(f"```text\n{prompt[:1500]}\n...\n```")

```text
You answer questions about the Nobel Prizes using only the passages below.
Cite the passages you rely on by id, like [1]. If the answer is not in
them, say you don't know rather than guessing.

<passages>
[1] (Francis Halzen > Nobel Prize in Physics)
In 2026, Halzen was awarded the Nobel Prize in Physics "for decisive contributions to the IceCube Neutrino Observatory and the discovery of high-energy neutrinos of astrophysical origin." In announcing the award, the Royal Swedish Academy of Sciences credited Halzen's scientific leadership in developing IceCube as an instrument for neutrino astronomy. It explained that high-energy neutrinos can point us to distant astrophysical particle accelerators, because neutrinos travel over cosmic distances without changing direction or losing energy.

[2] (List of Nobel laureates in Physics > Statistics)
The Nobel Prize in Physics has been awarded to 230 individuals as of 2026. The first prize in physics was awarded in 1901 to Wilhelm Conrad Röntgen, of Germany, who received 150,782 SEK. John Bardeen is the only laureate to win the prize twice—in 1956 and 1972.
William Lawrence Bragg was the youngest Nobel laureate in physics; he won the prize in 1915 at the age of 25. He was also the youngest laureate for any Nobel prize until 2014 (when Malala Yousafzai won the Nobel Peace Prize at the age of 17). The oldest Nobel Prize laureate in physics was Arthur Ashkin who was 96 years old when he was awarded the prize in 2018.
Only five women have 
...
```

In [14]:
t0 = time.perf_counter()
answer, live = ask(prompt, CACHED["rag-2026"], max_tokens=400)
seconds = time.perf_counter() - t0

sources = "\n".join(f"{n}. {label(parents[p])}: {parents[p]['url']}"
                    for n, (p, _) in enumerate(top, start=1))
display_md(
    f"**Q:** {QUESTION}\n\n**A:** {answer.strip()}\n\n"
    f"*{'Generated in ' + format(seconds, '.1f') + ' s' if live else 'Cached answer (no AWS access)'}*\n\n"
    f"**Sources**\n\n{sources}"
)

**Q:** Who won the Nobel Prize in Physics in 2026?

**A:** Francis Halzen won the 2026 Nobel Prize in Physics "for decisive contributions to the IceCube Neutrino Observatory and the discovery of high-energy neutrinos of astrophysical origin" [1][3][5]. The Royal Swedish Academy of Sciences credited his scientific leadership in developing IceCube as an instrument for neutrino astronomy [1]. He is listed as being from the United States and Belgium [3].

*Generated in 2.5 s*

**Sources**

1. Francis Halzen > Nobel Prize in Physics: https://en.wikipedia.org/w/index.php?oldid=1381473977
2. List of Nobel laureates in Physics > Statistics: https://en.wikipedia.org/w/index.php?oldid=1381322896
3. List of Nobel laureates in Physics > Laureates: https://en.wikipedia.org/w/index.php?oldid=1381322896
4. Geoffrey Hinton > Honours and awards: https://en.wikipedia.org/w/index.php?oldid=1380282819
5. 2026 Nobel Prizes > Prizes: https://en.wikipedia.org/w/index.php?oldid=1380470315

The same model that did not know the answer a few cells ago now gives it, quotes the prize
citation, and points to the passages it used. The links lead to the exact Wikipedia revisions
in the index, so a reader can check every claim.

<div style="border-left: 4px solid #C8102E; background: rgba(200, 16, 46, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#C8102E; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Retrieval supplies the facts, the generator supplies the answer</strong><br>
The model's weights stayed the same. What changed is what we put in front of it: five passages, found by two engines, fused, and reranked. That is the whole idea of RAG, and everything else in the pipeline exists to make those five passages the right ones.
</div>

## A Modular, Tunable System

Every stage is optional. To see what each one buys, we run four variants on 14 questions with
known answers. Each question comes with a short answer string, such as "Halzen" for the 2026
physics prize. A passage counts as containing the answer if its text contains that string; the
match ignores case and skips the "Title > Section" header, so an article name alone does not
count. For every question and variant we record $r_q$, the rank of the first passage in the
ranking that contains the answer, and report three numbers over the questions $Q$:

- **Answer at rank 1:** the number of questions with $r_q = 1$; the generator gets the answer
  in its first passage.
- **Answer in top 5:** the number of questions with $r_q \le 5$; the answer reaches the five
  passages we hand to the generator.
- **MRR**, the mean reciprocal rank of Chapter 2 (Performance Evaluation), with one relevant
  passage per question, the first that contains the answer:

$$
\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{r_q}
$$

A question whose answer appears nowhere in the ranking adds 0. MRR rewards a rank of 1 much
more than a rank of 3 (1 against 0.33).

We also measure the retrieval time per question, with generation excluded.

In [15]:
QUESTIONS = [
    ("Who won the Nobel Prize in Physics in 2026?", "Halzen"),
    ("Who won the Nobel Prize in Physics in 2025?", "Martinis"),
    ("Who received the 2024 physics Nobel together with Geoffrey Hinton?", "Hopfield"),
    ("Which chemists shared the 2025 Nobel Prize in Chemistry?", "Yaghi"),
    ("Who got the 2025 Nobel Prize for medicine?", "Sakaguchi"),
    ("Who was awarded the Nobel Peace Prize in 2025?", "Machado"),
    ("Which writer received the 2025 Nobel Prize in Literature?", "Krasznahorkai"),
    ("Who won the literature Nobel in 2026?", "Anne Carson"),
    ("Which economists were honoured in 2025 for explaining innovation-driven growth?", "Mokyr"),
    ("Where does Michel Devoret work?", "Yale"),
    ("Which company's quantum hardware effort did John Martinis lead?", "Google"),
    ("At which university is John Clarke a professor emeritus?", "Berkeley"),
    ("What is a transmon?", "charge qubit"),
    ("How much prize money does a Nobel laureate receive?", "kronor"),
]

def first_hit(ranking, answer):
    """Rank of the first parent whose body (without header) contains the answer, or None."""
    return next((r + 1 for r, (p, _) in enumerate(ranking)
                 if answer.lower() in parents[p]["body"].lower()), None)

def variants(question):
    """Rankings and cumulative retrieval time of the four pipeline variants."""
    t0 = time.perf_counter(); s = sparse.search(question, 50)
    t1 = time.perf_counter(); d = resolve_to_parents(dense.search(question, 50), children)
    t2 = time.perf_counter(); f = rrf([s, d])
    t3 = time.perf_counter(); r = rerank(question, f[:30], parents, cross_encoder, top_k=30)
    t4 = time.perf_counter()
    return {"BM25 only": (s, t1 - t0), "Dense only": (d, t2 - t1),
            "Hybrid (RRF)": (f, t3 - t0), "Hybrid + rerank": (r, t4 - t0)}

ranks, times = {}, {}
for q, a in QUESTIONS:
    for name, (ranking, seconds) in variants(q).items():
        ranks.setdefault(name, []).append(first_hit(ranking, a))
        times.setdefault(name, []).append(seconds)

rows = []
for name in ranks:
    rs = ranks[name]
    rows.append([name, sum(r == 1 for r in rs), sum(r is not None and r <= 5 for r in rs),
                 f"{sum(1 / r for r in rs if r) / len(rs):.3f}",
                 f"{1000 * sum(times[name]) / len(times[name]):.0f}"])
print_table(rows, ["Variant", "Answer at rank 1", "Answer in top 5", "MRR", "ms per question"])

| Variant         |   Answer at rank 1 |   Answer in top 5 |   MRR |   ms per question |
|:----------------|-------------------:|------------------:|------:|------------------:|
| BM25 only       |                  7 |                14 | 0.732 |                 6 |
| Dense only      |                 10 |                13 | 0.784 |                 7 |
| Hybrid (RRF)    |                  7 |                14 | 0.66  |                13 |
| Hybrid + rerank |                 11 |                14 | 0.881 |               591 |

Read the table as the book's speed-for-quality trade:

- **Easy questions do not need the full chain.** In the top 5, every variant finds nearly every
  answer. For a factual lookup, BM25 alone is fast and good enough.
- **Fusion is not automatically better.** RRF rewards agreement, so a passage that one engine
  ranks first and the other misses can drop below passages both rank moderately. Fusion widens
  the candidate pool; it does not pick the best candidate.
- **The reranker earns its cost at the top.** It puts the answer first most often, which is what
  matters when only a handful of passages reach the generator, and it is by far the slowest
  stage. That is why production systems route easy queries down a short path.

With 14 questions, one question is worth 0.07 of a rate: read the pattern, not the decimals.

## Building the Pipeline in Haystack

The book builds the hybrid core of the pipeline with Haystack. The Sentence-Transformers
components and the model generators live in separate integration packages
(`haystack_integrations`). As in the book, both retrievers search the same chunks, here the
parents. The generator is Claude on Bedrock.

In [16]:
from haystack import Document, Pipeline
from haystack.components.builders import ChatPromptBuilder
from haystack.components.joiners import DocumentJoiner
from haystack.components.retrievers.in_memory import (
    InMemoryBM25Retriever,
    InMemoryEmbeddingRetriever,
)
from haystack.components.writers import DocumentWriter
from haystack.dataclasses import ChatMessage
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder,
)
from haystack_integrations.components.rankers.sentence_transformers import (
    SentenceTransformersSimilarityRanker,
)
from haystack_integrations.components.generators.amazon_bedrock import (
    AmazonBedrockChatGenerator,
)
from shared.llm import MODEL_ID, AWS_REGION

The **indexing pipeline** embeds the documents and writes them to an in-memory store:

In [17]:
store = InMemoryDocumentStore()
documents = [Document(content=p["text"], meta={"title": p["title"], "section": p["section"]})
             for p in parents]

indexing = Pipeline()
indexing.add_component("embedder", SentenceTransformersDocumentEmbedder(
    model=f"sentence-transformers/{EMBED_MODEL}", progress_bar=False))
indexing.add_component("writer", DocumentWriter(document_store=store))
indexing.connect("embedder", "writer")

indexing.run({"embedder": {"documents": documents}})
display_md(f"The store holds **{store.count_documents()} documents** with embeddings.")

The store holds **634 documents** with embeddings.

The **query pipeline** is the graph of the book: BM25 and dense retrieval in parallel, fused
by reciprocal rank fusion, reranked by a cross-encoder, and passed to a generator.

In [18]:
template = [ChatMessage.from_user("""Answer the question using only the passages below.
{% for doc in documents %}[{{ loop.index }}] {{ doc.content }}
{% endfor %}
Question: {{ query }}""")]

rag = Pipeline()
rag.add_component("bm25", InMemoryBM25Retriever(document_store=store, top_k=50))
rag.add_component("embedder", SentenceTransformersTextEmbedder(
    model=f"sentence-transformers/{EMBED_MODEL}", progress_bar=False))
rag.add_component("dense", InMemoryEmbeddingRetriever(document_store=store, top_k=50))
rag.add_component("join", DocumentJoiner(join_mode="reciprocal_rank_fusion"))
rag.add_component("rerank", SentenceTransformersSimilarityRanker(model=RERANK_MODEL, top_k=5,
                                                                 scale_score=False))
rag.add_component("prompt", ChatPromptBuilder(template=template,
                                              required_variables=["query", "documents"]))
rag.add_component("llm", AmazonBedrockChatGenerator(model=MODEL_ID, aws_region_name=AWS_REGION))

rag.connect("embedder.embedding", "dense.query_embedding")
rag.connect("bm25", "join")
rag.connect("dense", "join")
rag.connect("join", "rerank")
rag.connect("rerank", "prompt.documents")
rag.connect("prompt", "llm");

In [19]:
try:
    result = rag.run({
        "bm25": {"query": QUESTION},
        "embedder": {"text": QUESTION},
        "rerank": {"query": QUESTION},
        "prompt": {"query": QUESTION},
    }, include_outputs_from={"rerank"})
    reply, passages = result["llm"]["replies"][0].text, result["rerank"]["documents"]
    note = ""
except Exception as e:                      # no AWS credentials or no model access
    reply, passages = CACHED["haystack-2026"], []
    note = f"\n\n*Cached answer: the generator could not run ({type(e).__name__}).*"

display_md(f"**Q:** {QUESTION}\n\n**A:** {reply.strip()}{note}")
if passages:
    print_table([[r + 1, d.meta["title"], d.meta["section"], f"{d.score:.3f}"]
                 for r, d in enumerate(passages)], ["Rank", "Title", "Section", "Score"])

**Q:** Who won the Nobel Prize in Physics in 2026?

**A:** Francis Halzen won the 2026 Nobel Prize in Physics. According to the passages, the prize was awarded "for decisive contributions to the IceCube Neutrino Observatory and the discovery of high-energy neutrinos of astrophysical origin." The Royal Swedish Academy of Sciences credited his scientific leadership in developing IceCube as an instrument for neutrino astronomy.

|   Rank | Title                              | Section                |   Score |
|-------:|:-----------------------------------|:-----------------------|--------:|
|      1 | Francis Halzen                     | Nobel Prize in Physics |   9.538 |
|      2 | List of Nobel laureates in Physics | Statistics             |   8.599 |
|      3 | List of Nobel laureates in Physics | Laureates              |   7.637 |
|      4 | Geoffrey Hinton                    | Honours and awards     |   7.41  |
|      5 | 2026 Nobel Prizes                  | Prizes                 |   6.827 |

The Haystack pipeline gives the same kind of answer from a few lines of wiring, and every
component can be swapped for another: a production store, a different reranker, another model.
Its five passages and their cross-encoder scores are the same as in our hand-built pipeline:
the reranker decides the final order, and both candidate pools contain the same top passages.
Two differences from our hand-built version are worth noticing:

- **One chunk level.** Both retrievers search the 2000-character parents, so the bi-encoder sees
  only their first 256 tokens. Haystack's `HierarchicalDocumentSplitter` and
  `AutoMergingRetriever` implement the two-level arrangement we built by hand.
- **A fixed graph.** The pipeline encodes one way of answering one kind of question. A question
  with two parts, such as "who won, and where do they work?", needs a second retrieval that
  depends on the first answer, which this graph cannot express. That is the starting point of
  *Agentic RAG*.

## Summary

| Stage | Here | What it contributes |
|---|---|---|
| Two-level chunking | parents of 2000, children of 400 characters | sharp dense matching, enough context for the answer |
| Sparse retrieval | BM25 over parents | exact terms, years, names; fast |
| Dense retrieval | all-MiniLM-L6-v2 over children, resolved to parents | meaning beyond shared words |
| Hybrid fusion | reciprocal rank fusion | one candidate pool from two engines |
| Reranking | ms-marco-MiniLM cross-encoder, top 30 to top 5 | the right passages first, at the highest cost |
| Generation | Claude with the grounded-answer prompt | a direct answer with citations |

A language model cannot answer what happened after its training. Retrieval brings the facts in
at query time, and the pipeline's stages decide which few passages the model gets to see.

## Try it yourself

1. Change `QUESTION` to a two-part question, for example "Who won the 2026 Nobel Prize in
   Physics, and where does he work?" Do the five passages contain both answers?
2. In the variants comparison, replace RRF's constant 60 by 1 (`rrf(..., k=1)` in
   `variants`). How do rank-1 counts and MRR of the hybrid change?
3. Rerank 10 instead of 30 candidates. How much faster is it, and which questions lose their
   answer from the top 5?